# 03 — Le performance (tentativo 13)

Sulle testate valutate (fold per gruppo, etichetta NewsGuard 2026): il **classificatore a feature** (voto degli
articoli) e **ModernBERT** (media delle probabilità degli articoli), con le baseline e il riferimento **"solo
orientamento + paese"** (regressione logistica sui soli metadati NewsGuard di orientamento e paese, stessi fold e pesi:
quanto si indovina senza leggere il testo). Si salvano:

- `testate_probabilita.csv`: per ogni testata valutata, fold, livello vero, orientamento, paese e, per ogni modello,
  le **probabilità medie delle 5 classi** e il livello previsto;
- `intervalli.csv`: le metriche con gli intervalli al 95% (bootstrap per gruppo, 2000 ricampionamenti);
- `per_fold.csv`: le metriche di **ogni fold**, per ogni modello;
- `per_orientamento.csv`: le metriche dentro ogni orientamento (Right, Left, nessuno);
- `differenze.csv`: ModernBERT − feature, sugli stessi ricampionamenti.

In [ ]:
import json

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from comune import CLASSI, LIVELLI, RESULTS, VERSIONI, metriche_di, per_testata, pesi_bilanciati

B = 2000
OUT = RESULTS / "confronto"; OUT.mkdir(parents=True, exist_ok=True)
t = pd.read_csv(VERSIONI / "fold.csv").set_index("domain")
val = t[t.valutazione].sort_index()
METR = ["f1_macro", "accuratezza", "tolerant_1", "bin_accuratezza"]
tab = val[["fold_gruppo", "gruppo", "label", "orientamento", "paese"]].rename(columns={"label": "y_true"})
PRED = {}
f = RESULTS / "feature_tuned_2026_fold_gruppo_articoli.csv"
if f.exists():
    a = pd.read_csv(f)
    p = a.groupby("domain")[CLASSI].mean().reindex(val.index)
    for k, c in enumerate(CLASSI):
        tab[f"feature {c}"] = p[c]
    tab["feature previsto (voto)"] = per_testata(a).y_pred.reindex(val.index)
    PRED["feature"] = tab["feature previsto (voto)"].to_numpy().astype(int)
f = RESULTS / "transformer" / "modernbert_2026_s42" / "articoli.csv"
if f.exists() and (f.parent / "FATTO").exists():
    p = pd.read_csv(f).groupby("domain")[CLASSI].mean().reindex(val.index)
    for c in CLASSI:
        tab[f"ModernBERT {c}"] = p[c]
    tab["ModernBERT previsto (media)"] = p.to_numpy().argmax(1)
    PRED["ModernBERT"] = tab["ModernBERT previsto (media)"].to_numpy()
# riferimento senza testo: solo orientamento e paese
X = pd.get_dummies(t[["orientamento", "paese"]]).astype(float); y = t.label.to_numpy(); meta = pd.Series(-1, index=t.index)
for fo in sorted(t.fold_gruppo.unique()):
    tr, te = (t.fold_gruppo != fo).to_numpy(), ((t.fold_gruppo == fo) & t.valutazione).to_numpy()
    m = LogisticRegression(max_iter=2000).fit(X[tr], y[tr], sample_weight=pesi_bilanciati(y[tr], t.peso.to_numpy()[tr]))
    meta[te] = m.predict(X[te])
tab["solo orientamento + paese, previsto"] = meta.reindex(val.index)
PRED["solo orientamento + paese"] = tab["solo orientamento + paese, previsto"].to_numpy().astype(int)
tab.to_csv(OUT / "testate_probabilita.csv")
print(f"{len(val)} testate valutate; modelli: {list(PRED)}")

In [ ]:
def quattro(y, p):
    M = np.bincount(y * 5 + p, minlength=25).reshape(5, 5)
    tp = np.diag(M); den = M.sum(0) + M.sum(1)
    f1 = np.divide(2 * tp, den, out=np.zeros(5), where=den > 0).mean()
    d = np.abs(y - p)
    return [f1, (d == 0).mean(), (d <= 1).mean(), ((y >= 2) == (p >= 2)).mean()]


Y = val.label.to_numpy().astype(int); g = val.gruppo.to_numpy()
gruppi = np.unique(g); pos = {x: np.where(g == x)[0] for x in gruppi}
rng = np.random.default_rng(0)
camp = [np.concatenate([pos[x] for x in rng.choice(gruppi, len(gruppi))]) for _ in range(B)]
BS = {k: np.array([quattro(Y[i], p[i]) for i in camp]) for k, p in PRED.items()}
righe = []
for k, p in PRED.items():
    m = metriche_di(Y, p); assert np.allclose([m[x] for x in METR], quattro(Y, p))
    righe.append({"modello": k, **{x: m[x] for x in METR}, **{f"{x} 2.5%": np.percentile(BS[k][:, j], 2.5) for j, x in enumerate(METR)},
                  **{f"{x} 97.5%": np.percentile(BS[k][:, j], 97.5) for j, x in enumerate(METR)}})
for nome in ["maggioranza", "a_caso"]:
    f = RESULTS / f"baseline_{nome}_2026_fold_gruppo_metriche.json"
    if f.exists():
        m = json.load(open(f))["testata_voto"]; righe.append({"modello": f"baseline: {nome.replace('_', ' ')}", **{x: m[x] for x in METR}})
pd.DataFrame(righe).to_csv(OUT / "intervalli.csv", index=False)
diff = []
for a, b in [("ModernBERT", "feature"), ("feature", "solo orientamento + paese"), ("ModernBERT", "solo orientamento + paese")]:
    if a in PRED and b in PRED:
        d = BS[a] - BS[b]; ma, mb = metriche_di(Y, PRED[a]), metriche_di(Y, PRED[b])
        diff.append({"confronto": f"{a} − {b}", **{f"Δ {x}": ma[x] - mb[x] for x in METR}, **{f"Δ {x} 2.5%": np.percentile(d[:, j], 2.5) for j, x in enumerate(METR)},
                     **{f"Δ {x} 97.5%": np.percentile(d[:, j], 97.5) for j, x in enumerate(METR)}})
pd.DataFrame(diff).to_csv(OUT / "differenze.csv", index=False)
display(pd.DataFrame(righe).set_index("modello")[METR].round(3)); pd.DataFrame(diff).set_index("confronto")[[f"Δ {x}" for x in METR]].round(3)

In [ ]:
righe = []
for k, p in PRED.items():
    for fo in sorted(val.fold_gruppo.unique()):
        s = (val.fold_gruppo == fo).to_numpy(); m = metriche_di(Y[s], p[s])
        righe.append({"modello": k, "fold": fo, "testate": int(s.sum()), **{x: m[x] for x in METR}})
per_fold = pd.DataFrame(righe); per_fold.to_csv(OUT / "per_fold.csv", index=False)
righe = []
for k, p in PRED.items():
    for o in ["Right", "Left", "nessuno"]:
        s = (val.orientamento == o).to_numpy(); m = metriche_di(Y[s], p[s])
        righe.append({"modello": k, "orientamento": o, "testate": int(s.sum()), **{x: m[x] for x in METR},
                      **{f"testate {l}": int((Y[s] == i).sum()) for i, l in enumerate(LIVELLI)}})
per_or = pd.DataFrame(righe); per_or.to_csv(OUT / "per_orientamento.csv", index=False)
(OUT / "aggiornato.txt").write_text(pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"))
display(per_fold.pivot(index="fold", columns="modello", values="f1_macro").round(3))
per_or.pivot(index="orientamento", columns="modello", values="f1_macro").round(3)